# Lab 10 - Environment Setup (run once per environment)

**WHAT:** Create the isolated Lab 10 schema and its four Delta tables, seed the CDF-enabled source from Neon PostgreSQL through Lakehouse Federation, and initialise the CDC checkpoint.

**WHY:** The CDC notebooks expect these objects to exist. In Personal DEV they were created interactively; this notebook makes any new environment reproducible.

**INPUT:** `catalog`, `schema`, `foreign_catalog` parameters; the foreign catalog must already exist.

**OUTPUT:** `customers_delta` (CDF on), `customers_target`, `customer_change_history`, `cdc_checkpoint`.

**RERUN SAFETY:** Idempotent. It never drops, truncates, overwrites or reseeds. Objects that exist are left alone; an inconsistent state stops the notebook instead of being repaired by guesswork.

**FIRST CDC RUN:** The checkpoint is set to version 0 (the table creation), so the first run of `02_incremental_cdc_sync` processes the seed as a real CDC batch.

In [ ]:
# WHAT: Read deployment parameters. Defaults are the Personal DEV values.
# WHY: The same notebook runs in Personal DEV and Azure PROD without edits;
#      the Azure Job passes its own catalog, schema and foreign catalog.
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "lab_10_federation_cdc")
dbutils.widgets.text("foreign_catalog", "neon_postgres_conn_catalog")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
FOREIGN_CATALOG = dbutils.widgets.get("foreign_catalog").strip()

# Identifiers are interpolated into SQL, so only plain names are accepted.
for name, value in {"catalog": CATALOG, "schema": SCHEMA, "foreign_catalog": FOREIGN_CATALOG}.items():
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), f"Invalid {name}: {value!r}"

BASE = f"{CATALOG}.{SCHEMA}"
print(f"Delta schema: {BASE}" + f" | foreign catalog: {FOREIGN_CATALOG}")

SOURCE = f"{BASE}.customers_delta"
TARGET = f"{BASE}.customers_target"
HISTORY = f"{BASE}.customer_change_history"
CHECKPOINT = f"{BASE}.cdc_checkpoint"
PIPELINE = "customers_delta_to_target"

In [ ]:
# WHAT: Create the schema and tables only if they do not exist.
# WHY: Column types match Personal DEV exactly, so both environments behave the same.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BASE} COMMENT 'Lab 10 Federation & CDC - isolated environment'")

COLUMNS = """
    customer_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(150),
    city VARCHAR(60),
    country VARCHAR(60),
    signup_date DATE
"""

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SOURCE} ({COLUMNS})
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql(f"CREATE TABLE IF NOT EXISTS {TARGET} ({COLUMNS})")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HISTORY} ({COLUMNS},
        _change_type STRING,
        _commit_version BIGINT,
        _commit_timestamp TIMESTAMP)
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHECKPOINT} (
        pipeline_name STRING,
        last_processed_version BIGINT,
        processed_at TIMESTAMP)
""")

# A pre-existing source without CDF cannot feed table_changes; stop rather than alter it.
cdf = spark.sql(f"SHOW TBLPROPERTIES {SOURCE} ('delta.enableChangeDataFeed')").first()["value"]
assert cdf == "true", f"Change Data Feed is not enabled on {SOURCE}"
print("Schema and tables are present; CDF is enabled on the source.")

In [ ]:
# WHAT: Seed the source from Neon and initialise the checkpoint - only for a brand-new environment.
# WHY: Seeding through the foreign catalog keeps Lakehouse Federation as the integration method.
count = lambda table: spark.sql(f"SELECT COUNT(*) AS n FROM {table}").first()["n"]

checkpoint_rows = spark.sql(f"""
    SELECT last_processed_version FROM {CHECKPOINT}
    WHERE pipeline_name = '{PIPELINE}'
""").collect()
source_rows, target_rows, history_rows = count(SOURCE), count(TARGET), count(HISTORY)

if len(checkpoint_rows) == 1:
    print(f"Already initialised (checkpoint {checkpoint_rows[0][0]}); nothing changed.")
elif len(checkpoint_rows) > 1:
    raise AssertionError("More than one checkpoint row; fix manually before running CDC.")
elif target_rows or history_rows:
    raise AssertionError(
        "Target or history has data but no checkpoint exists; refusing to guess a version."
    )
else:
    # CDF has been on since the CREATE (version 0), so every later commit is captured.
    first = spark.sql(f"DESCRIBE HISTORY {SOURCE}").orderBy("version").first()
    assert first["version"] == 0 and first["operation"].startswith("CREATE"), (
        "Source history does not start with its creation; refusing to initialise."
    )
    if source_rows == 0:
        spark.sql(f"""
            INSERT INTO {SOURCE}
            SELECT
                CAST(customer_id AS INT),
                CAST(first_name AS VARCHAR(50)),
                CAST(last_name AS VARCHAR(50)),
                CAST(email AS VARCHAR(150)),
                CAST(city AS VARCHAR(60)),
                CAST(country AS VARCHAR(60)),
                CAST(signup_date AS DATE)
            FROM {FOREIGN_CATALOG}.public.lab10_customers
        """)
        print(f"Seeded {count(SOURCE)} customers from {FOREIGN_CATALOG} via Federation.")
    spark.sql(f"""
        INSERT INTO {CHECKPOINT}
        VALUES ('{PIPELINE}', CAST(0 AS BIGINT), current_timestamp())
    """)
    print("Checkpoint initialised at version 0; the next CDC run processes the seed.")

print(f"Source rows: {count(SOURCE)} | target rows: {count(TARGET)} | history rows: {count(HISTORY)}")